# 03 — Climate Data Preprocessing

Extracts daily Berlin climate values from ERA5 and CAMS netCDF files, derives additional indices, and merges with the health data to produce the master analysis file.

**Inputs**:
- `data/raw/era5/era5_berlin_YYYY_MM.nc` — hourly ERA5 per month (from `src/download_era5.py`)
- `data/raw/cams/cams_berlin_YYYY.nc` — daily CAMS (from notebook 01)
- `data/processed/daily_city_health.csv` — from notebook 02

**Outputs**:
- `data/processed/daily_climate.csv` — daily ERA5 + CAMS + derived indices + rolling features
- `data/processed/daily_merged.csv` — master analysis file (climate + health + confounders)

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

ERA5_DIR = Path('../data/raw/era5')
CAMS_DIR = Path('../data/raw/cams')
PROC_DIR = Path('../data/processed')

# Berlin reference point (city centre)
BERLIN_LAT = 52.52
BERLIN_LON = 13.405

print('Libraries loaded')

## 1. Load ERA5 — Extract Berlin Point, Resample to Daily

In [ ]:
import xarray as xr
from pathlib import Path

era5_files = sorted(ERA5_DIR.glob("era5_berlin_*.nc"))

valid_files = []
bad_files = []

for f in era5_files:
    try:
        with xr.open_dataset(f, engine="netcdf4") as ds:
            pass
        valid_files.append(f)
    except Exception as e:
        bad_files.append((f, str(e)))

print(f"Valid files: {len(valid_files)}")
print(f"Bad files: {len(bad_files)}")

for f, err in bad_files:
    print(f"\nBAD FILE: {f}")
    print(err)

In [ ]:
ds_era5 = xr.open_mfdataset(
    valid_files,
    combine="by_coords",
    engine="netcdf4"
)

print("ERA5 variables:", list(ds_era5.data_vars))
print("Time range:", str(ds_era5.time.values[0])[:10], "→", str(ds_era5.time.values[-1])[:10])
print("Grid:", dict(ds_era5.sizes))

In [ ]:
if len(era5_files) > 0:
    # Extract nearest grid point to Berlin centre
    pt = ds_era5.sel(latitude=BERLIN_LAT, longitude=BERLIN_LON, method='nearest')

    # Variable name mapping (ERA5 short names vary by download method)
    var_map = {
        't2m':  'temp_2m_k',
        'd2m':  'dewpoint_2m_k',
        'tp':   'precip_m',
        'u10':  'u_wind',
        'v10':  'v_wind',
        'ssrd': 'solar_rad',
    }
    pt = pt.rename({k: v for k, v in var_map.items() if k in pt})

    # Resample to daily aggregates
    daily_era5 = xr.Dataset()
    if 'temp_2m_k'    in pt: daily_era5['temp_mean'] = (pt['temp_2m_k'] - 273.15).resample(time='1D').mean()
    if 'temp_2m_k'    in pt: daily_era5['temp_max']  = (pt['temp_2m_k'] - 273.15).resample(time='1D').max()
    if 'temp_2m_k'    in pt: daily_era5['temp_min']  = (pt['temp_2m_k'] - 273.15).resample(time='1D').min()
    if 'dewpoint_2m_k' in pt: daily_era5['dewpoint'] = (pt['dewpoint_2m_k'] - 273.15).resample(time='1D').mean()
    if 'precip_m'     in pt: daily_era5['precip_mm'] = (pt['precip_m'] * 1000).resample(time='1D').sum()
    if 'u_wind'       in pt: daily_era5['u_wind']    = pt['u_wind'].resample(time='1D').mean()
    if 'v_wind'       in pt: daily_era5['v_wind']    = pt['v_wind'].resample(time='1D').mean()
    if 'solar_rad'    in pt: daily_era5['solar_rad'] = pt['solar_rad'].resample(time='1D').sum()

    era5_df = daily_era5.to_dataframe().reset_index()
    era5_df = era5_df.rename(columns={'time': 'date'})
    era5_df['date'] = pd.to_datetime(era5_df['date']).dt.normalize()
    print(f'ERA5 daily shape: {era5_df.shape}')
    era5_df.head(3)

## 2. Derive Additional Indices

In [ ]:
if len(era5_files) > 0:
    # Wind speed magnitude
    if 'u_wind' in era5_df.columns and 'v_wind' in era5_df.columns:
        era5_df['wind_speed'] = np.sqrt(era5_df['u_wind']**2 + era5_df['v_wind']**2)

    # Relative humidity from temperature and dewpoint (Magnus formula)
    if 'temp_mean' in era5_df.columns and 'dewpoint' in era5_df.columns:
        T = era5_df['temp_mean']
        Td = era5_df['dewpoint']
        era5_df['relative_humidity'] = 100 * np.exp((17.625 * Td) / (243.04 + Td)) / \
                                              np.exp((17.625 * T)  / (243.04 + T))

    # Heat index (Rothfusz equation) — meaningful only when T >= 27°C and RH >= 40%
    # For cold periods, use temp_mean directly
    if 'temp_max' in era5_df.columns and 'relative_humidity' in era5_df.columns:
        T  = era5_df['temp_max']
        RH = era5_df['relative_humidity']
        HI = (-8.78469475556 +
               1.61139411   * T +
               2.33854883889 * RH +
              -0.14611605   * T * RH +
              -0.012308094  * T**2 +
              -0.0164248277778 * RH**2 +
               0.002211732  * T**2 * RH +
               0.00072546   * T * RH**2 +
              -0.000003582  * T**2 * RH**2)
        # Heat index only valid for hot/humid conditions; otherwise use temp_max
        era5_df['heat_index'] = np.where(T >= 27, HI, T)

    # Cold stress: Wind chill (valid when T <= 10°C and wind > 1.3 m/s)
    if 'temp_min' in era5_df.columns and 'wind_speed' in era5_df.columns:
        T = era5_df['temp_min']
        V = era5_df['wind_speed']
        WC = 13.12 + 0.6215*T - 11.37*(V*3.6)**0.16 + 0.3965*T*(V*3.6)**0.16
        era5_df['wind_chill'] = np.where((T <= 10) & (V > 1.3), WC, T)

    print('Derived columns:', ['wind_speed','relative_humidity','heat_index','wind_chill'])
    era5_df[['date','temp_mean','temp_max','temp_min','relative_humidity',
              'heat_index','wind_chill','precip_mm']].describe().round(2)

## 3. Load CAMS Air Quality Data

In [ ]:
cams_files = sorted(CAMS_DIR.glob('cams_berlin_*.nc'))
print(f'Found {len(cams_files)} CAMS files')

if len(cams_files) == 0:
    print('No CAMS files yet. Run notebook 01 after setting up ADS credentials.')
    print('Creating empty CAMS placeholder DataFrame.')
    cams_df = pd.DataFrame(columns=['date','pm25','pm10','no2','o3','so2'])
else:
    ds_cams = xr.open_mfdataset(cams_files, combine='by_coords')
    print('CAMS variables:', list(ds_cams.data_vars))

    pt_cams = ds_cams.sel(latitude=BERLIN_LAT, longitude=BERLIN_LON, method='nearest')

    # CAMS variable name mapping (may vary by version)
    cams_var_map = {
        'pm2p5': 'pm25',
        'pm10':  'pm10',
        'no2':   'no2',
        'o3':    'o3',
        'so2':   'so2',
        'pm2_5': 'pm25',  # alternate naming
    }

    daily_cams = xr.Dataset()
    for src, dst in cams_var_map.items():
        if src in pt_cams:
            # Convert from kg/m³ to µg/m³ (multiply by 1e9)
            daily_cams[dst] = (pt_cams[src] * 1e9).resample(time='1D').mean()

    cams_df = daily_cams.to_dataframe().reset_index()
    cams_df = cams_df.rename(columns={'time': 'date'})
    cams_df['date'] = pd.to_datetime(cams_df['date']).dt.normalize()
    print(f'CAMS daily shape: {cams_df.shape}')
    cams_df.head(3)

## 4. Merge ERA5 + CAMS, Add Rolling Features

In [ ]:
if len(era5_files) > 0:
    climate_df = era5_df.merge(cams_df, on='date', how='left')
else:
    climate_df = cams_df.copy()

climate_df = climate_df.sort_values('date').reset_index(drop=True)

# Rolling features — shift(1) to use *prior* days only (no data leakage into same day)
def rolling_lag(series, window):
    return series.shift(1).rolling(window, min_periods=window//2).mean()

if 'temp_mean' in climate_df.columns:
    climate_df['temp_mean_3d']  = rolling_lag(climate_df['temp_mean'], 3)
    climate_df['temp_mean_7d']  = rolling_lag(climate_df['temp_mean'], 7)
    climate_df['temp_max_3d']   = rolling_lag(climate_df['temp_max'],  3)

if 'pm25' in climate_df.columns:
    climate_df['pm25_mean_3d']  = rolling_lag(climate_df['pm25'], 3)
    climate_df['pm25_mean_7d']  = rolling_lag(climate_df['pm25'], 7)

if 'o3' in climate_df.columns:
    climate_df['o3_mean_3d']    = rolling_lag(climate_df['o3'],  3)

if 'precip_mm' in climate_df.columns:
    climate_df['precip_3d_sum'] = climate_df['precip_mm'].shift(1).rolling(3, min_periods=1).sum()

print(f'Climate dataset: {climate_df.shape}')
print(f'Date range: {climate_df.date.min()} → {climate_df.date.max()}')
climate_df.head(3)

## 5. Save Climate CSV

In [ ]:
climate_out = PROC_DIR / 'daily_climate.csv'
climate_df.to_csv(climate_out, index=False)
print(f'Saved: {climate_out}  ({climate_df.shape[0]} rows, {climate_df.shape[1]} cols)')

## 6. Merge with Health Data → Master Analysis File

In [ ]:
health_path = PROC_DIR / 'daily_city_health.csv'
if not health_path.exists():
    print('Health data not found — run notebook 02 first')
else:
    health_df = pd.read_csv(health_path, parse_dates=['date'])
    print(f'Health data: {health_df.shape}')

    merged = health_df.merge(climate_df, on='date', how='left')

    climate_cols = [c for c in climate_df.columns if c != 'date']
    missing_climate = merged[climate_cols].isnull().mean()
    pct_missing = (missing_climate > 0).sum()
    print(f'Climate columns with any missing values: {pct_missing} / {len(climate_cols)}')

    merged_out = PROC_DIR / 'daily_merged.csv'
    merged.sort_values('date').to_csv(merged_out, index=False)
    print(f'\nMaster file saved: {merged_out}')
    print(f'Shape: {merged.shape[0]} rows × {merged.shape[1]} cols')
    merged.head(3)

## 7. Diagnostic Plots

In [ ]:
if len(era5_files) > 0 and health_path.exists():
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

    # Temperature
    ax = axes[0]
    ax.fill_between(merged['date'], merged['temp_min'], merged['temp_max'],
                     alpha=0.2, color='tomato')
    ax.plot(merged['date'], merged['temp_mean'], color='tomato', lw=1, label='T mean')
    ax.set_ylabel('Temperature (°C)')
    ax.set_title('Daily Temperature Range — Berlin (ERA5)')
    ax.legend()

    # EMS critical count
    ax = axes[1]
    if 'mission_count_ems_critical' in merged.columns:
        ax.plot(merged['date'], merged['mission_count_ems_critical'],
                color='steelblue', lw=0.5, alpha=0.5)
        ax.plot(merged['date'],
                merged['mission_count_ems_critical'].rolling(14, center=True).mean(),
                color='steelblue', lw=2, label='14-day mean')
    ax.set_ylabel('Daily count')
    ax.set_title('Critical EMS Missions (city-wide)')
    ax.legend()

    # Breathing difficulty vs temperature (scatter proxy)
    ax = axes[2]
    if 'code_06_breathing' in merged.columns and 'temp_max' in merged.columns:
        ax.scatter(merged['temp_max'], merged['code_06_breathing'],
                   s=4, alpha=0.3, color='darkorange')
        ax.set_xlabel('Daily max temperature (°C)')
        ax.set_ylabel('Daily breathing calls')
        ax.set_title('Breathing difficulties vs Max Temperature (raw, no confounders)')

    fig.tight_layout()
    plt.savefig(PROC_DIR / 'climate_health_diagnostic.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: data/processed/climate_health_diagnostic.png')

In [ ]:
# Correlation heatmap (raw, before deconfounding — for visual exploration only)
if len(era5_files) > 0 and health_path.exists():
    import seaborn as sns

    health_cols  = [c for c in merged.columns if c.startswith('code_')]
    climate_cols = ['temp_mean','temp_max','temp_min','relative_humidity',
                    'heat_index','wind_chill','precip_mm','wind_speed']
    climate_cols = [c for c in climate_cols if c in merged.columns]

    corr = merged[health_cols + climate_cols].corr(method='spearman')
    sub  = corr.loc[health_cols, climate_cols]

    fig, ax = plt.subplots(figsize=(len(climate_cols)*1.2 + 2, len(health_cols)*0.7 + 2))
    sns.heatmap(sub, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                vmin=-1, vmax=1, ax=ax, linewidths=0.3)
    ax.set_title('Spearman correlation: health outcomes vs climate variables (raw, uncontrolled)')
    fig.tight_layout()
    plt.savefig(PROC_DIR / 'raw_correlation_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Note: These raw correlations include seasonal confounding — interpret with caution.')
    print('Controlled estimates will follow in notebook 04.')